# Feature Engineering

- начинаем с удаления признаков с очень большим количеством пропусков (count NaN > 75%)
- заполняем NaN у оставшихся признаков средним значением
- на V-признаках делаем PCA
- вычисляем TransactionDay = TransactionDT / (24 * 60 * 60), далее используя card1, addr1, d1n (transactionDay - d1) вычисляем uid
- на основе uid группируем признаки и считаем их std, mean
- создаем другие новые признаки D4n, D10n, D15n

In [2]:
import cudf as cd
import cupy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import numpy as np
import pandas as pd

## 1. Очистка данных и загрузка
На этапе EDA мы проанализировали пропуски. Признаки, где доля `NaN` превышает 75%, не несут ценности и только создают шум. Мы загружаем их список из `nans.json` и сразу исключаем при чтении CSV с помощью `cudf` для экономии RAM и VRAM.

In [3]:
import json

with open("nans.json", "r", encoding="utf-8") as f:
    nans_groups = json.load(f)

In [4]:
cols = [508189, 449124, 460110, 450721, 450909, 508589, 508595, 528353, 528588, 525823, 515614, 551623, 517353, 453249, 552913]
d = [j for row in [nans_groups[f'{i}'] for i in cols] for j in row]

In [5]:
cols_t = ['TransactionID', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11',
       'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8',
       'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4',
       'M5', 'M6', 'M7', 'M8', 'M9']
cols_v = ['V'+str(x) for x in range(1,340)]; types_v = {}
for c in cols_v: types_v[c] = 'float32' # конвертируем для экономии памяти

train_trans = cd.read_csv(r'../data/raw/train_transaction.csv', usecols=cols_t+['isFraud']+cols_v, dtype=types_v).drop(columns=d)

test_trans = cd.read_csv(r'../data/raw/test_transaction.csv', usecols=cols_t+cols_v, dtype=types_v).drop(columns=d)

## 2. Frequency Encoding
Для категориальных признаков используем частотное кодирование. Это позволяет деревьям решений лучше улавливать редкие категории без раздувания размерности, как при One-Hot Encoding.

In [6]:
for col in train_trans.select_dtypes(include=['str']).columns.to_list():
    freq_map = train_trans[col].value_counts(normalize=True).to_dict()
    train_trans[f'{col}_encoded'] = train_trans[col].map(freq_map)
    test_trans[f'{col}_encoded'] = test_trans[col].map(freq_map)

train_trans = train_trans.drop(columns=train_trans.select_dtypes(include=['str']).columns.to_list())
test_trans = test_trans.drop(columns=test_trans.select_dtypes(include=['str']).columns.to_list())

## 3. Обработка пропусков
Оставшиеся пропуски в числовых признаках заполняем средним значением. Для более сложных признаков (например, `D-features`) в будущем можно использовать групповое заполнение по `uid`, но для базового пайплайна `mean` дает стабильный результат.

In [7]:
train_trans = train_trans.fillna(train_trans.mean(numeric_only=True))
test_trans = test_trans.fillna(test_trans.mean(numeric_only=True))

## 4. Снижение размерности (PCA для V-features)
Признаки `V1 - V339` содержат огромное количество скоррелированных данных и шумов. Мы разбиваем их на подгруппы и применяем PCA с использованием `cuML`. Это позволяет сжать сотни колонок в несколько главных компонент, сохранив долю объясненной дисперсии.

In [8]:
from cuml.decomposition import PCA
from cuml.preprocessing import StandardScaler

cols_w_d1 = ['V281', 'V282', 'V283', 'V288', 'V289', 'V296', 'V300', 'V301', 'V313', 'V314', 'V315']
cols = ['279287', '76073', '168969', '77096', '89164', '314', '12']

features = [f'v_pca_{j}' for j in range(3)]
    
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_trans[cols_w_d1])
test_scaled = scaler.transform(test_trans[cols_w_d1])

# PCA (сокращаем до 3 главных компонент)
pca = PCA(n_components=3)

train_trans[features] = pca.fit_transform(train_scaled)
test_trans[features] = pca.transform(test_scaled)

train_trans = train_trans.drop(columns=cols_w_d1)
test_trans = test_trans.drop(columns=cols_w_d1)

for i, col in enumerate(cols):
    features = [f'v{i}_pca_{j}' for j in range(3)]
    
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_trans[nans_groups[col]])
    test_scaled = scaler.transform(test_trans[nans_groups[col]])
    
    # PCA (сокращаем до 3 главных компонент)
    pca = PCA(n_components=3)
    
    train_trans[features] = pca.fit_transform(train_scaled)
    test_trans[features] = pca.transform(test_scaled)

    train_trans = train_trans.drop(columns=nans_groups[col])
    test_trans = test_trans.drop(columns=nans_groups[col])
    
    print(f'Доля объясненной дисперсии {i}-ой группы:', pca.explained_variance_ratio_)

Доля объясненной дисперсии 0-ой группы: 0    0.381695
1    0.168242
2    0.125581
dtype: float64
Доля объясненной дисперсии 1-ой группы: 0    0.415292
1    0.120791
2    0.086664
dtype: float32
Доля объясненной дисперсии 2-ой группы: 0    0.412156
1    0.174613
2    0.100988
dtype: float32
Доля объясненной дисперсии 3-ой группы: 0    0.443739
1    0.127080
2    0.082708
dtype: float32
Доля объясненной дисперсии 4-ой группы: 0    0.404442
1    0.128079
2    0.096059
dtype: float32
Доля объясненной дисперсии 5-ой группы: 0    0.363400
1    0.179170
2    0.070698
dtype: float32
Доля объясненной дисперсии 6-ой группы: 0    0.439539
1    0.109495
2    0.095835
dtype: float32


## 5. Генерация `uid` и поведенческих фичей 
* `TransactionDay` и сдвиги `D1n`, `D4n`, `D15n` помогают нормализовать время.
* `uid` — кастомный ключ (`card1` + `addr1` + `D1n`), который позволяет идентифицировать одного пользователя даже при смене аккаунта.
* На основе `uid` мы считаем агрегации (mean, std) для суммы транзакции и дистанций. 

In [9]:
train_trans['D1n'] = np.floor(train_trans.TransactionDT / (24*60*60)) - train_trans.D1
train_trans['D4n'] = np.floor(train_trans.TransactionDT / (24*60*60)) - train_trans.D4
train_trans['D10n'] = np.floor(train_trans.TransactionDT / (24*60*60)) - train_trans.D10
train_trans['D15n'] = np.floor(train_trans.TransactionDT / (24*60*60)) - train_trans.D15
train_trans['uid'] = train_trans.card1.astype(str)+'_'+train_trans.addr1.astype(str)+'_'+train_trans.D1n.astype(str)

test_trans['D1n'] = np.floor(test_trans.TransactionDT / (24*60*60)) - test_trans.D1
test_trans['D4n'] = np.floor(test_trans.TransactionDT / (24*60*60)) - test_trans.D4
test_trans['D10n'] = np.floor(test_trans.TransactionDT / (24*60*60)) - test_trans.D10
test_trans['D15n'] = np.floor(test_trans.TransactionDT / (24*60*60)) - test_trans.D15
test_trans['uid'] = test_trans.card1.astype(str)+'_'+test_trans.addr1.astype(str)+'_'+test_trans.D1n.astype(str)

In [10]:
for col in ['D4n', 'D10n', 'D15n', 'TransactionAmt', 'dist1']: 
    train_trans[f'{col}_uid_mean'] = train_trans.groupby('uid')[col].transform('mean')
    train_trans[f'{col}_uid_std'] = train_trans.groupby('uid')[col].transform('std')
    test_trans[f'{col}_uid_mean'] = test_trans.groupby('uid')[col].transform('mean')
    test_trans[f'{col}_uid_std'] = test_trans.groupby('uid')[col].transform('std')

    train_trans[f'{col}_uid_std'].fillna(-1, inplace=True)
    test_trans[f'{col}_uid_std'].fillna(-1, inplace=True)

Перед обучением моделей удаляем признак `uid` для избежания утечки данных при предсказывании на тесте.

In [11]:
train_trans = train_trans.drop(columns='uid')
test_trans = test_trans.drop(columns='uid')

## 6. Обучение и GPU-тюнинг
Используем Optuna для оптимизации гиперпараметров. 
Обучение XGBoost, LightGBM и CatBoost происходит с использованием `device='cuda'` / `task_type='GPU'`, что сокращает время итераций с часов до минут.

P.S.: обучить LightGBM на CUDA не удалось ввиду технических ошибок. Решение для устранения проблемы пока не найдено. Также не удалось запустить CatBoost в Optuna ввиду неправильного разделения потоков, решение пока не найдено.

In [12]:
import optuna
from sklearn.metrics import roc_auc_score
from cuml.model_selection import train_test_split
import xgboost as xgb

X_train, X_val, y_train, y_val = train_test_split(train_trans.drop(columns='isFraud'), train_trans['isFraud'], test_size=0.2)

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1200, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
    }
    model = xgb.XGBClassifier(**params, tree_method='hist', device='cuda', random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val.to_numpy(), preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)
print(f'XGB Best parameters: {study.best_params}, Best ROC-AUC CV value: {study.best_value}')

[I 2026-08-24 20:02:35,201] A new study created in memory with name: no-name-ef29fc42-1faa-4d89-b56d-27ad6a40673a
[I 2026-08-24 20:02:41,961] Trial 0 finished with value: 0.9353259577698261 and parameters: {'max_depth': 5, 'n_estimators': 1050, 'learning_rate': 0.016552967966956555}. Best is trial 0 with value: 0.9353259577698261.
[I 2026-08-24 20:02:53,919] Trial 1 finished with value: 0.9623791487470876 and parameters: {'max_depth': 10, 'n_estimators': 550, 'learning_rate': 0.011297979230996732}. Best is trial 1 with value: 0.9623791487470876.
[I 2026-08-24 20:02:55,528] Trial 2 finished with value: 0.9479870247145806 and parameters: {'max_depth': 4, 'n_estimators': 300, 'learning_rate': 0.1961936713706609}. Best is trial 1 with value: 0.9623791487470876.
[I 2026-08-24 20:03:07,969] Trial 3 finished with value: 0.9770692720631816 and parameters: {'max_depth': 10, 'n_estimators': 850, 'learning_rate': 0.15181651919488523}. Best is trial 3 with value: 0.9770692720631816.
[I 2026-08-24 

XGB Best parameters: {'max_depth': 11, 'n_estimators': 650, 'learning_rate': 0.08242569290499799}, Best ROC-AUC CV value: 0.9790548004848416


In [13]:
results = []

results.append({
        'Model': 'XGBoost',
        'ROC-AUC': study.best_value,
    })

In [14]:
import lightgbm as lgb

# def objective_lgb(trial):
#     params = {
#         'max_depth': trial.suggest_int('max_depth', 2, 15),
#         'n_estimators': trial.suggest_int('n_estimators', 300, 1500, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#     }
model_lgb = lgb.LGBMClassifier(device='cpu', random_state=42, verbose=-1, max_depth=10, n_estimators=900, learning_rate=0.09)
model_lgb.fit(X_train.to_numpy(), y_train.to_numpy())
preds_lgb = model_lgb.predict_proba(X_val.to_numpy())[:, 1]
lgb_score = roc_auc_score(y_val.to_numpy(), preds_lgb)
    # return roc_auc_score(y_val.to_numpy(), preds_lgb)

# study_lgb = optuna.create_study(direction='maximize')
# study_lgb.optimize(objective_lgb, n_trials=100)
# print(f'LGBM Best parameters: {study_lgb.best_params}, Best ROC-AUC value: {study_lgb.best_value}')
print(lgb_score)

0.9704649172226942


In [15]:
results.append({
        'Model': 'Light GBM',
        'ROC-AUC': lgb_score,
    })

In [16]:
import catboost as cb

# def objective_cb(trial):
#     params = {
#         'max_depth': trial.suggest_int('max_depth', 2, 15),
#         'n_estimators': trial.suggest_int('n_estimators', 50, 1000, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#     }
cb_model = cb.CatBoostClassifier(task_type="GPU", random_state=42, verbose=0, max_depth=10, n_estimators=900, learning_rate=0.09)
cb_model.fit(X_train.to_pandas(), y_train.to_pandas())
cb_preds = cb_model.predict_proba(X_val.to_pandas())[:, 1]
cb_score = roc_auc_score(y_val.to_numpy(), cb_preds)
# return roc_auc_score(y_val.to_numpy(), preds)

# study_cb = optuna.create_study(direction='maximize')
# study_cb.optimize(objective_cb, n_trials=50, n_jobs=1)
# print(f'CatBoost Best parameters: {study_cb.best_params}, Best ROC-AUC value: {study_cb.best_value}')
print(cb_score)

0.9669887087145963


In [17]:
results.append({
        'Model': 'CatBoost',
        'ROC-AUC': cb_score,
    })

In [18]:
cd.DataFrame(results)

,Model,ROC-AUC
0,XGBoost,0.979055
1,Light GBM,0.970465
2,CatBoost,0.966989


## 7. Blending
Простое усреднение предсказаний позволяет сгладить переобучение отдельных моделей и повысить итоговый ROC-AUC на лидерборде. При этом, взвешенное усреднение дало немного лучший результат на LB, ввиду лучшего roc_auc_score у XGBoost.

In [19]:
xgb_model = xgb.XGBClassifier(max_depth=10, n_estimators=800, learning_rate=0.105, tree_method='hist', device='cuda', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict_proba(X_val)[:, 1]

blend_preds = xgb_preds * 0.5 + preds_lgb * 0.25 + cb_preds * 0.25
blend_score = roc_auc_score(y_val.to_numpy(), blend_preds)

In [20]:
results.append({
        'Model': 'Blending',
        'ROC-AUC': blend_score,
    })

In [21]:
results[3]['ROC-AUC'] = blend_score
cd.DataFrame(results)

,Model,ROC-AUC
0,XGBoost,0.979055
1,Light GBM,0.970465
2,CatBoost,0.966989
3,Blending,0.976632


In [25]:
sample_df = pd.read_csv(r'../data/final/sample_submission.csv')
sample_df['isFraud'] = xgb_model.predict_proba(test_trans)[:, 1] * 0.5 + model_lgb.predict_proba(test_trans)[:, 1] * 0.25 + cb_model.predict_proba(test_trans.to_numpy())[:, 1] * 0.25

sample_df.to_csv('../data/final/blending.csv', index=False)